# Chatbot Monitoring with the Galileo Python SDK

This notebook keeps the Galileo SDK calls inline so the setup and instrumentation are visible as executable code blocks. The goal is to show the actual sequence of SDK operations with minimal helper abstraction.


In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv

for root in [Path.cwd(), Path.cwd().parent]:
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))

env_candidates = [Path.cwd() / '.env', Path.cwd().parent / '.env']
for candidate in env_candidates:
    if candidate.exists():
        load_dotenv(candidate)
        ENV_FILE = candidate
        break
else:
    raise FileNotFoundError('Could not find .env in the repo root or current directory')

print(f'Loaded credentials from {ENV_FILE}')


In [ ]:
from galileo import GalileoLogger, GalileoMetrics, Message, MessageRole, galileo_context
from galileo.config import GalileoPythonConfig
from galileo.log_streams import create_log_stream, enable_metrics
from galileo.openai import openai
from galileo.projects import delete_project

PROJECT_NAME = 'GalileoEval_S1_Chatbot'
LOG_STREAM_NAME = 'chatbot-monitoring'
MODEL = 'gpt-4o-mini'

PROJECT_NAME, LOG_STREAM_NAME, MODEL


## 1. Initialize Galileo

Initialize the Galileo context, make sure the log stream exists, and print the console links if Galileo returns IDs.


In [ ]:
galileo_context.init(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)

try:
    create_log_stream(name=LOG_STREAM_NAME, project_name=PROJECT_NAME)
except Exception:
    pass

config = GalileoPythonConfig.get()
logger = galileo_context.get_logger_instance()
project_id = getattr(logger, 'project_id', None)
log_stream_id = getattr(logger, 'log_stream_id', None)

if project_id and log_stream_id:
    project_url = f'{config.console_url}project/{project_id}'
    log_stream_url = f'{project_url}/log-streams/{log_stream_id}'
    print(project_url)
    print(log_stream_url)
else:
    print('Galileo context initialized')


## 2. Create an OpenAI Client Through Galileo

The Galileo OpenAI wrapper instruments the calls automatically, so the SDK usage stays close to normal OpenAI client code.


In [ ]:
client = openai.OpenAI()
client


## 3. Log a Single Chat Completion

The simplest pattern is: initialize context, make a model call, flush the Galileo context.


In [ ]:
galileo_context.init(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role': 'system', 'content': 'You are a helpful customer support agent for a software company.'},
        {'role': 'user', 'content': 'How do I reset my password?'},
    ],
    max_tokens=100,
)

galileo_context.flush()
response.choices[0].message.content


## 4. Log a Streaming Response

Streaming works the same way, except you accumulate deltas before flushing the context.


In [ ]:
galileo_context.init(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)

stream = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role': 'system', 'content': 'You are a helpful customer support agent.'},
        {'role': 'user', 'content': 'What are your business hours?'},
    ],
    max_tokens=80,
    stream=True,
)

chunks = []
for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        chunks.append(chunk.choices[0].delta.content)

galileo_context.flush()
''.join(chunks)


## 5. Track a Multi-Turn Session

Use `start_session()` before a sequence of related turns to group the conversation in Galileo.


In [ ]:
galileo_context.init(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)
galileo_context.start_session(name='customer-session-001')

conversation = [
    {'role': 'system', 'content': 'You are a customer support agent. Be concise.'},
    {'role': 'user', 'content': "I can't log into my account."},
]

first = client.chat.completions.create(model=MODEL, messages=conversation, max_tokens=80)
conversation.append({'role': 'assistant', 'content': first.choices[0].message.content})
conversation.append({'role': 'user', 'content': "I tried that but it still doesn't work."})

second = client.chat.completions.create(model=MODEL, messages=conversation, max_tokens=80)
conversation.append({'role': 'assistant', 'content': second.choices[0].message.content})
conversation.append({'role': 'user', 'content': 'Okay that worked, thank you!'})

final_response = client.chat.completions.create(model=MODEL, messages=conversation, max_tokens=80)

galileo_context.flush()
final_response.choices[0].message.content


## 6. Enable Quality and Safety Metrics

Metrics are configured on the log stream, then Galileo scores future traces for those dimensions.


In [ ]:
enable_metrics(
    project_name=PROJECT_NAME,
    log_stream_name=LOG_STREAM_NAME,
    metrics=[
        GalileoMetrics.correctness,
        GalileoMetrics.instruction_adherence,
        GalileoMetrics.input_pii,
        GalileoMetrics.output_pii,
        GalileoMetrics.input_toxicity,
        GalileoMetrics.output_toxicity,
    ],
)


## 7. Send a Manual Tagged Trace

You can also bypass the automatic OpenAI wrapper and log a trace explicitly, which is useful when you want full control over tags, span boundaries, and token accounting.


In [ ]:
logger = GalileoLogger(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)
logger.start_trace(
    input='Customer asking about refund policy',
    tags={'customer_tier': 'premium', 'topic': 'refunds', 'env': 'staging'},
)
logger.add_llm_span(
    input=[
        Message(role=MessageRole.system, content='You are a support agent.'),
        Message(role=MessageRole.user, content='What is your refund policy?'),
    ],
    output=Message(
        role=MessageRole.assistant,
        content='Our refund policy allows returns within 30 days of purchase.',
    ),
    model=MODEL,
    num_input_tokens=25,
    num_output_tokens=15,
    tags={'response_type': 'policy_info'},
)
logger.conclude(output='Our refund policy allows returns within 30 days of purchase.')
logger.flush()


## 8. Clean Up the Demo Project

Run this only when you want to remove the Galileo project created by the notebook.


In [ ]:
delete_project(name=PROJECT_NAME)
